In [ ]:
import math
import random

class Neuron:
    def __init__(self, numinputs):
        self.numin = numinputs
        self.w = [0.01 * random.random() for _ in range(numinputs)]
        self.b = 0.01 * random.random()

    def forwardPass(self, inputs):
        z = self.b
        for i in range(self.numin):
            z += self.w[i] * inputs[i]
        out = 1.0 / (1.0 + math.exp(-z))
        return out

    def backwardPass(self, prev, inputs, pred, lr):
        # prev is dL/d(out)
        dz = pred * (1.0 - pred)
        dLdz = prev * dz
        for i in range(self.numin):
            dw = dLdz * inputs[i]
            self.w[i] -= lr * dw
        self.b -= lr * dLdz

    def feedBackward(self, prev, pred):
        # returns dL/d(inputs)
        dz = pred * (1.0 - pred)
        dLdz = prev * dz
        return [dLdz * self.w[i] for i in range(self.numin)]


class Layer:
    def __init__(self, neuronCount, numinputs):
        self.neurons = [Neuron(numinputs) for _ in range(neuronCount)]
        self.inputs = [0.0] * numinputs
        self.outputs = [0.0] * neuronCount

    def forwardPass(self, inputs):
        self.inputs = list(inputs)
        for i, n in enumerate(self.neurons):
            self.outputs[i] = n.forwardPass(self.inputs)
        return self.outputs[:]

    def backwardPass(self, chain, lr):
        # chain length equals neuronCount
        for i, n in enumerate(self.neurons):
            n.backwardPass(chain[i], self.inputs, self.outputs[i], lr)

    def feedBackward(self, chain):
        # add dL/d(inputs) across neurons
        derivatives = [0.0] * len(self.inputs)
        for i, n in enumerate(self.neurons):
            outD = n.feedBackward(chain[i], self.outputs[i])
            for j in range(len(self.inputs)):
                derivatives[j] += outD[j]
        return derivatives


class NeuralNet:
    def __init__(self, layer_sizes):
        # layer_sizes: input_dim - hidden - output_dim
        self.layers = [Layer(out_dim, in_dim) for in_dim, out_dim in zip(layer_sizes[:-1], layer_sizes[1:])]

    def forwardPass(self, x):
        for layer in self.layers:
            x = layer.forwardPass(x)
        return x

    def backwardPass(self, dL_dout, lr):
        chain = list(dL_dout)
        for layer in reversed(self.layers):
            layer.backwardPass(chain, lr)
            chain = layer.feedBackward(chain)

    def mse(self, pred, target):
        return 0.5 * sum((p - t) ** 2 for p, t in zip(pred, target))

    def mse_grad(self, pred, target):
        return [p - t for p, t in zip(pred, target)]

    def train(self, X, Y, lr=0.01, epochs=1000, log_every=100):
        data = list(zip(X, Y))
        loss = 0.0
        for epoch in range(epochs):
            for x,y in data:
                yhat = self.forwardPass(x)
                loss += self.mse(yhat, y)
                self.backwardPass(self.mse_grad(yhat, y), lr)
            if epoch%log_every==0:
                print(f"epoch = {epoch} loss = {loss / len(data):.6f}")

    def predict(self, X):
        return [self.forwardPass(x) for x in X]

    def evaluate(self, X, Y):
        return sum(self.mse(p, y) for p, y in zip(self.predict(X), Y)) / len(X)


xs = [float(x) for x in range(10)]
ys = [x / 10.0 for x in xs]
X = [[x] for x in xs]
Y = [[y] for y in ys]

net = NeuralNet([1, 100, 1])
net.train(X, Y, lr=0.01, epochs=5000, log_every=200)

print("\nPredictions:")
for x, y, p in zip(xs, ys, net.predict(X)):
    print(f"x = {int(x):2d}  target = {y:.2f}  pred = {p[0]:.4f}")

print(f"\nAverage loss: {net.evaluate(X, Y):.6f}")
        

epoch = 0 loss = 0.047240
epoch = 200 loss = 8.451990
epoch = 400 loss = 16.740979
epoch = 600 loss = 24.429286
epoch = 800 loss = 29.726736
epoch = 1000 loss = 31.898284
epoch = 1200 loss = 32.676468
epoch = 1400 loss = 33.025618
epoch = 1600 loss = 33.229207
epoch = 1800 loss = 33.376363
epoch = 2000 loss = 33.499494
epoch = 2200 loss = 33.611731
epoch = 2400 loss = 33.718796
epoch = 2600 loss = 33.823312
epoch = 2800 loss = 33.926518
epoch = 3000 loss = 34.029014
epoch = 3200 loss = 34.131095
epoch = 3400 loss = 34.232908
epoch = 3600 loss = 34.334526
epoch = 3800 loss = 34.435988
epoch = 4000 loss = 34.537312
epoch = 4200 loss = 34.638510
epoch = 4400 loss = 34.739587
epoch = 4600 loss = 34.840549
epoch = 4800 loss = 34.941398

Predictions:
x =  0  target = 0.00  pred = 0.0589
x =  1  target = 0.10  pred = 0.1030
x =  2  target = 0.20  pred = 0.1725
x =  3  target = 0.30  pred = 0.2705
x =  4  target = 0.40  pred = 0.3904
x =  5  target = 0.50  pred = 0.5164
x =  6  target = 0.60  

In [13]:
words = open('datasets/names.txt','r').read().lower().splitlines()

# Build vocab with '.' as 0
chars = sorted(list(set(''.join(words))))
vocabSize = len(chars) + 1
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

print("vocabSize:", vocabSize)
print("itos (first 10):", {k:itos[k] for k in sorted(itos)[:10]})

blockSize = 1  # context length in characters (here just previous single char)

def oneHot(num, size):
    v = [0]*size
    v[num] = 1
    return v

def buildDataset(words):
    X, Y = [], []
    for w in words:
        context = [0]*blockSize  # start with <.> token index 0
        for ch in w + '.':
            ix = stoi[ch]
            # Input is the one-hot of the current context (size vocabSize*blockSize)
            # Since blockSize=1, this is just one one-hot vector of length vocabSize.
            # If you later increase blockSize, you can concatenate multiple one-hots.
            x_vec = []
            for c in context:
                x_vec.extend(oneHot(c, vocabSize))
            y_vec = oneHot(ix, vocabSize)
            X.append(x_vec)
            Y.append(y_vec)
            # slide the window
            context = context[1:] + [ix]
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = buildDataset(words[:n1])
Xdev, Ydev = buildDataset(words[n1:n2])
Xte, Yte = buildDataset(words[n2:])

input_dim = vocabSize * blockSize
output_dim = vocabSize

print("Sample X shape:", len(Xtr))
print("Sample Y length:", len(Ytr))
print(Xtr[0],Ytr[0])

vocabSize: 27
itos (first 10): {0: '.', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i'}
Sample X shape: 182625
Sample Y length: 182625
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]


In [7]:
net = NeuralNet([input_dim, 100, 50, output_dim])

# Train on training set
for _ in range(10):
    net.train(Xtr, Ytr, lr=0.1, batch=32)

# Evaluate
tr_loss = net.evaluate(Xtr, Ytr)
dev_loss = net.evaluate(Xdev, Ydev)
te_loss = net.evaluate(Xte, Yte)
print(f"Train loss: {tr_loss:.6f}")
print(f"Dev loss:   {dev_loss:.6f}")
print(f"Test loss:  {te_loss:.6f}")

# Example predictions: take argmax to get next-char prediction
def argmax(v):
    best_i, best_v = 0, v[0]
    for i, val in enumerate(v):
        if val > best_v:
            best_v, best_i = val, i
    return best_i

def predict_next_char(prev_ix):
    # prev_ix is an int index for previous character (0 for start)
    x = oneHot(prev_ix, vocabSize)
    yhat = net.forwardPass(x)
    return argmax(yhat)

print("Example next-char index from start:", predict_next_char(0), "->", itos[predict_next_char(0)])

165954
113103
8753
92857
164434
22740
152833
27072
123415
1948
41980
178443
70261
48781
62322
112134
62997
115719
65628
173430
56366
121240
65374
140014
11864
17663
57109
19725
137764
68889
50535
75298
loss = 0.000189
19679
49398
90297
176675
126234
31278
114672
147203
37457
108541
32593
17138
28482
75222
120918
100074
44715
6598
116715
53732
66935
120917
141211
126216
60019
90127
81798
31730
73549
84835
91485
109460
loss = 0.000089
80037
118505
50854
16832
170893
25614
103921
134179
139930
85672
155308
46169
106043
105492
60658
124103
167008
4937
3616
142983
169639
137018
158444
70394
58613
143869
50946
11887
32656
78174
143394
175084
loss = 0.000084
95920
124235
132933
27275
106063
97916
113657
2562
136074
138809
12383
177687
69363
17964
133236
179250
162744
9895
175896
34041
73741
74536
105420
48387
145685
138112
44688
113306
17197
122496
53798
68761
loss = 0.000084
181529
98968
8908
142896
119933
57024
113752
52852
20497
68657
147842
51156
180760
24559
135339
22382
49803
118337
350

KeyboardInterrupt: 